In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Tuple
import re
from sentence_transformers import SentenceTransformer

## Тестовая выборка запросов

In [2]:
with open("data/test_queries.json", "r", encoding="utf-8") as f:
    queries = json.load(f)

df_queries = pd.DataFrame(queries)
print(f"Загружено запросов: {len(df_queries)}")
df_queries.head()

Загружено запросов: 32


,id,query,type,relevant_docs
0,1,Какие этапы проходит исходный код C++ от препр...,conceptual,[Универ/5 семестр/Программирование на языке C+...
1,2,"Где искать причину, если программа не линкуетс...",troubleshooting,[Универ/5 семестр/Программирование на языке C+...
2,3,Почему макрос для квадрата неправильно обрабат...,troubleshooting,[Универ/5 семестр/Программирование на языке C+...
3,4,Как передать аргументы командной строки в функ...,simple,[Универ/5 семестр/Программирование на языке C+...
4,5,"Чем связаны понятия Data Science, Big Data и d...",comparison,[Универ/5 семестр/Инструменты анализа данных.md]


In [ ]:
def load_corpus(docs_path: str) -> List[Dict]:
    corpus = []
    docs_dir = Path(docs_path)
    
    for md_file in docs_dir.rglob("*.md"):
        try:
            content = md_file.read_text(encoding="utf-8")
            
            content = re.sub(r"```[\s\S]*?```", "", content)
            content = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", content)
            content = re.sub(r"#+\s*", "", content)
            content = re.sub(r"\*\*([^*]+)\*\*", r"\1", content)
            content = re.sub(r"\*([^*]+)\*", r"\1", content)
            content = re.sub(r"\s+", " ", content).strip()
            
            corpus.append({
                "doc_id": str(md_file),
                "filename": str(md_file.relative_to(docs_dir)),
                "content": content,
                "length": len(content)
            })
        except Exception as e:
            print(f"Ошибка при загрузке {md_file}: {e}")
    
    print(f"Загружено документов: {len(corpus)}")
    print(f"Общий объём текста: {sum(doc['length'] for doc in corpus):,} символов")
    return corpus

In [4]:
corpus = load_corpus("../../data")

Загружено документов: 133
Общий объём текста: 2,483,152 символов


In [ ]:
def build_tfidf_index(corpus: List[Dict]) -> Tuple[TfidfVectorizer, np.ndarray]:
    """Строит TF-IDF матрицу для корпуса документов"""
    documents = [doc["content"] for doc in corpus]
    
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=10000,
        ngram_range=(1, 2),
        min_df=2,  
        max_df=0.95
    )
    
    tfidf_matrix = vectorizer.fit_transform(documents)
    
    print(f"Размер словаря: {len(vectorizer.vocabulary_):,} слов")
    print(f"Размер матрицы: {tfidf_matrix.shape}")
    print(f"Плотность матрицы: {tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.4f}")
    
    return vectorizer, tfidf_matrix

In [6]:
vectorizer, tfidf_matrix = build_tfidf_index(corpus)

Размер словаря: 10,000 слов
Размер матрицы: (133, 10000)
Плотность матрицы: 0.0550


In [ ]:
def search_tfidf(
    query: str,
    vectorizer: TfidfVectorizer,
    tfidf_matrix: np.ndarray,
    corpus: List[Dict],
    top_k: int = 10
) -> List[Dict]:
    query_vector = vectorizer.transform([query])
    
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    top_indices = similarities.argsort()[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        if similarities[idx] > 0:
            results.append({
                "doc_id": corpus[idx]["doc_id"],
                "filename": corpus[idx]["filename"],
                "score": float(similarities[idx]),
                "content_preview": corpus[idx]["content"][:200] + "..."
            })
    
    return results

In [8]:
with open("data/test_queries.json", "r", encoding="utf-8") as f:
    test_queries = json.load(f)

In [ ]:
tfidf_results = {}

for query_data in test_queries:
    query_id = query_data["id"]
    query_text = query_data["query"]
    
    results = search_tfidf(
        query=query_text,
        vectorizer=vectorizer,
        tfidf_matrix=tfidf_matrix,
        corpus=corpus,
        top_k=20
    )
    
    tfidf_results[query_id] = results
    
    print(f"\nЗапрос #{query_id}: {query_text}")
    print("Топ-3 результата:")
    for i, result in enumerate(results[:3], 1):
        print(f"  {i}. {result['filename']} (score: {result['score']:.4f})")


Запрос #1: Какие этапы проходит исходный код C++ от препроцессинга до запуска?
Топ-3 результата:
  1. Универ/7 семестр/WEB приложения.md (score: 0.0771)
  2. Универ/экзамены, зачёты, контрольные/3 семестр/Линукс.md (score: 0.0758)
  3. Универ/6 семестр/домашка/Фронт доклад.md (score: 0.0709)

Запрос #2: Где искать причину, если программа не линкуется из-за повторного определения функции?
Топ-3 результата:
  1. Мысли/Ловец снов.md (score: 0.1650)
  2. Универ/экзамены, зачёты, контрольные/4 семестр/Оптимизация коллоквиум.md (score: 0.1407)
  3. Универ/экзамены, зачёты, контрольные/1 семестр/матан.md (score: 0.1310)

Запрос #3: Почему макрос для квадрата неправильно обрабатывает выражение вроде 5 + 1?
Топ-3 результата:
  1. Универ/5 семестр/дз/анализ данных/7.11.md (score: 0.1110)
  2. Универ/5 семестр/дз/анализ данных/28.11.md (score: 0.0696)
  3. Универ/5 семестр/дз/анализ данных/полное дз.md (score: 0.0530)

Запрос #4: Как передать аргументы командной строки в функцию main на C++?
Топ

In [10]:
df_results = []
for query_id, results in tfidf_results.items():
    query_text = next(q["query"] for q in test_queries if q["id"] == query_id)
    relevant_docs = next(q["relevant_docs"] for q in test_queries if q["id"] == query_id)
    
    for rank, result in enumerate(results, 1):
        is_relevant = result["filename"] in relevant_docs
        df_results.append({
            "query_id": query_id,
            "query": query_text,
            "rank": rank,
            "filename": result["filename"],
            "score": result["score"],
            "is_relevant": is_relevant,
            "relevant_docs": relevant_docs
        })

df_tfidf = pd.DataFrame(df_results)

In [ ]:
def calculate_metrics(results: Dict, test_queries: List[Dict], k_values: List[int] = [1, 3, 5, 10]):
    metrics = {f"Recall@{k}": [] for k in k_values}
    metrics[f"Precision@{k_values[2]}"] = []
    metrics["MRR"] = []
    
    for query_data in test_queries:
        query_id = query_data["id"]
        relevant_docs = set(query_data["relevant_docs"])
        
        if query_id not in results:
            continue
        
        retrieved = [r["filename"] for r in results[query_id]]
        
        for k in k_values:
            retrieved_k = set(retrieved[:k])
            recall = len(retrieved_k & relevant_docs) / len(relevant_docs) if relevant_docs else 0
            metrics[f"Recall@{k}"].append(recall)
        
        retrieved_k = set(retrieved[:5])
        precision = len(retrieved_k & relevant_docs) / 5
        metrics["Precision@5"].append(precision)
        
        mrr = 0
        for rank, doc in enumerate(retrieved, 1):
            if doc in relevant_docs:
                mrr = 1 / rank
                break
        metrics["MRR"].append(mrr)
    
    avg_metrics = {metric: np.mean(values) for metric, values in metrics.items()}
    return avg_metrics

In [12]:
tfidf_metrics = calculate_metrics(tfidf_results, test_queries)

In [13]:
for metric, value in tfidf_metrics.items():
    print(f"{metric:15s}: {value:.4f}")

Recall@1       : 0.1562
Recall@3       : 0.3125
Recall@5       : 0.4375
Recall@10      : 0.5625
Precision@5    : 0.0875
MRR            : 0.2759


## Семантический поиск: сравнение моделей

In [ ]:
model_configs = [
    {
        "name": "paraphrase-multilingual-MiniLM-L12-v2",
        "model_id": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "query_prefix": "",
        "document_prefix": "",
    },
    {
        "name": "multilingual-e5-small",
        "model_id": "intfloat/multilingual-e5-small",
        "query_prefix": "query: ",
        "document_prefix": "passage: ",
    },
    {
        "name": "distiluse-base-multilingual-cased-v2",
        "model_id": "sentence-transformers/distiluse-base-multilingual-cased-v2",
        "query_prefix": "",
        "document_prefix": "",
    },
]

In [ ]:
common_max_seq_length = 128
document_texts = [doc["content"] for doc in corpus]
query_texts = [item["query"] for item in test_queries]
semantic_embeddings = {}
semantic_results_by_model = {}
semantic_metrics_by_model = {}

In [ ]:
for config in model_configs:
    print(f"Загрузка модели: {config['name']}")
    model = SentenceTransformer(config["model_id"])
    model.max_seq_length = min(model.max_seq_length, common_max_seq_length)

    document_embeddings = model.encode(
        [config["document_prefix"] + text for text in document_texts],
        batch_size=16,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    query_embeddings = model.encode(
        [config["query_prefix"] + text for text in query_texts],
        batch_size=16,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    model_results = {}
    for query_index, query_data in enumerate(test_queries):
        scores = query_embeddings[query_index] @ document_embeddings.T
        top_indices = np.argsort(scores)[::-1][:20]
        model_results[query_data["id"]] = [
            {
                "filename": corpus[index]["filename"],
                "score": float(scores[index]),
            }
            for index in top_indices
        ]

    semantic_embeddings[config["name"]] = {
        "documents": document_embeddings,
        "queries": query_embeddings,
        "max_sequence_length": model.max_seq_length,
    }
    semantic_results_by_model[config["name"]] = model_results
    semantic_metrics_by_model[config["name"]] = calculate_metrics(
        model_results, test_queries
    )

    print(
        f"Эмбеддинги: документы {document_embeddings.shape}, "
        f"запросы {query_embeddings.shape}; "
        f"max_seq_length={model.max_seq_length}"
    )
    del model

In [ ]:
comparison_rows = [{"Model": "TF-IDF", **tfidf_metrics}]
comparison_rows.extend(
    {"Model": name, **metrics}
    for name, metrics in semantic_metrics_by_model.items()
)

model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("MRR", ascending=False)
    .reset_index(drop=True)
)
model_comparison.round(3)

Загрузка модели: paraphrase-multilingual-MiniLM-L12-v2


Batches: 100%|██████████| 2/2 [00:00<00:00, 56.09it/s]


Эмбеддинги: документы (133, 384), запросы (32, 384); max_seq_length=128
Загрузка модели: multilingual-e5-small


Batches: 100%|██████████| 2/2 [00:00<00:00, 60.18it/s]


Эмбеддинги: документы (133, 384), запросы (32, 384); max_seq_length=128
Загрузка модели: distiluse-base-multilingual-cased-v2


Batches: 100%|██████████| 2/2 [00:00<00:00, 41.39it/s]

Эмбеддинги: документы (133, 512), запросы (32, 512); max_seq_length=128


,Model,Recall@1,Recall@3,Recall@5,Recall@10,Precision@5,MRR
0,multilingual-e5-small,0.406,0.562,0.625,0.750,0.125,0.506
1,paraphrase-multilingual-MiniLM-L12-v2,0.344,0.469,0.719,0.781,0.144,0.471
2,distiluse-base-multilingual-cased-v2,0.250,0.438,0.469,0.531,0.094,0.356
3,TF-IDF,0.156,0.312,0.438,0.562,0.088,0.276


## Влияние разбиения документов на чанки

In [19]:
CHUNK_WORDS = 64
CHUNK_OVERLAP = 16
DOCS_ROOT = Path("../../data")


def clean_markdown_block(text: str) -> str:
    text = re.sub(r"```.*?```", " ", text, flags=re.DOTALL)
    text = re.sub(r"^#+\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"[`*_~]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def fixed_window_chunks(text: str) -> List[str]:
    words = text.split()
    step = CHUNK_WORDS - CHUNK_OVERLAP
    return [
        " ".join(words[start:start + CHUNK_WORDS])
        for start in range(0, len(words), step)
        if words[start:start + CHUNK_WORDS]
    ]


def paragraph_chunks(markdown_text: str) -> List[str]:
    source_without_code = re.sub(
        r"```.*?```", " ", markdown_text, flags=re.DOTALL
    )
    paragraphs = [
        clean_markdown_block(block)
        for block in re.split(r"\n\s*\n+", source_without_code)
    ]
    paragraphs = [paragraph for paragraph in paragraphs if paragraph]

    chunks = []
    current_words = []
    for paragraph in paragraphs:
        words = paragraph.split()
        if len(words) > CHUNK_WORDS:
            if current_words:
                chunks.append(" ".join(current_words))
                current_words = []
            chunks.extend(fixed_window_chunks(paragraph))
            continue

        if current_words and len(current_words) + len(words) > CHUNK_WORDS:
            chunks.append(" ".join(current_words))
            current_words = []
        current_words.extend(words)

    if current_words:
        chunks.append(" ".join(current_words))
    return chunks


chunk_strategies = {
    "fixed_64_overlap_16": fixed_window_chunks,
    "paragraph_aware_64": paragraph_chunks,
}
chunked_documents = {}
chunking_summary = []

for strategy_name, chunker in chunk_strategies.items():
    strategy_chunks = []
    for doc in corpus:
        if strategy_name == "fixed_64_overlap_16":
            source_text = doc["content"]
        else:
            source_path = DOCS_ROOT / doc["filename"]
            source_text = source_path.read_text(encoding="utf-8")
        chunks = chunker(source_text)
        if not chunks and doc["content"]:
            chunks = [doc["content"]]
        strategy_chunks.extend(
            {
                "filename": doc["filename"],
                "chunk_index": chunk_index,
                "text": chunk_text,
            }
            for chunk_index, chunk_text in enumerate(chunks)
        )

    chunked_documents[strategy_name] = strategy_chunks
    chunking_summary.append(
        {
            "Strategy": strategy_name,
            "Chunks": len(strategy_chunks),
            "Mean chunks/document": round(len(strategy_chunks) / len(corpus), 2),
        }
    )

chunking_summary_df = pd.DataFrame(chunking_summary)
chunking_summary_df

,Strategy,Chunks,Mean chunks/document
0,fixed_64_overlap_16,6009,45.18
1,paragraph_aware_64,6744,50.71


In [20]:
e5_config = next(
    config for config in model_configs if config["name"] == "multilingual-e5-small"
)
e5_model = SentenceTransformer(e5_config["model_id"])
e5_model.max_seq_length = common_max_seq_length
chunk_query_embeddings = e5_model.encode(
    [e5_config["query_prefix"] + query for query in query_texts],
    batch_size=16,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

document_indices = {doc["filename"]: index for index, doc in enumerate(corpus)}
chunking_results_by_strategy = {}
chunking_metrics_by_strategy = {}

for strategy_name, chunks in chunked_documents.items():
    print(f"Кодирование чанков: {strategy_name} ({len(chunks)} чанков)")
    chunk_embeddings = e5_model.encode(
        [e5_config["document_prefix"] + chunk["text"] for chunk in chunks],
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    chunk_scores = chunk_query_embeddings @ chunk_embeddings.T
    document_scores = np.full(
        (len(test_queries), len(corpus)), -np.inf, dtype=np.float32
    )

    for chunk_index, chunk in enumerate(chunks):
        document_index = document_indices[chunk["filename"]]
        document_scores[:, document_index] = np.maximum(
            document_scores[:, document_index], chunk_scores[:, chunk_index]
        )

    strategy_results = {}
    for query_index, query_data in enumerate(test_queries):
        top_indices = np.argsort(document_scores[query_index])[::-1][:20]
        strategy_results[query_data["id"]] = [
            {
                "filename": corpus[index]["filename"],
                "score": float(document_scores[query_index, index]),
            }
            for index in top_indices
        ]

    chunking_results_by_strategy[strategy_name] = strategy_results
    chunking_metrics_by_strategy[strategy_name] = calculate_metrics(
        strategy_results, test_queries
    )
    del chunk_embeddings

del e5_model

chunking_comparison_rows = [
    {
        "Strategy": "E5, whole document",
        **semantic_metrics_by_model["multilingual-e5-small"],
    }
]
chunking_comparison_rows.extend(
    {"Strategy": strategy_name, **metrics}
    for strategy_name, metrics in chunking_metrics_by_strategy.items()
)
chunking_comparison = (
    pd.DataFrame(chunking_comparison_rows)
    .sort_values("MRR", ascending=False)
    .reset_index(drop=True)
)
chunking_comparison.round(3)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18250.46it/s]


Кодирование чанков: fixed_64_overlap_16 (6009 чанков)


Batches: 100%|██████████| 188/188 [00:19<00:00,  9.47it/s]


Кодирование чанков: paragraph_aware_64 (6744 чанков)


Batches: 100%|██████████| 211/211 [00:21<00:00,  9.61it/s]


,Strategy,Recall@1,Recall@3,Recall@5,Recall@10,Precision@5,MRR
0,paragraph_aware_64,0.438,0.812,0.875,0.969,0.175,0.650
1,fixed_64_overlap_16,0.406,0.750,0.875,0.969,0.175,0.609
2,"E5, whole document",0.406,0.562,0.625,0.750,0.125,0.506


## Итоговое сравнение и рекомендации

In [22]:
all_variant_rows = [{"Вариант": "TF-IDF", **tfidf_metrics}]
all_variant_rows.extend(
    {"Вариант": name, **metrics}
    for name, metrics in semantic_metrics_by_model.items()
)
all_variant_rows.extend(
    {"Вариант": f"E5 + {name}", **metrics}
    for name, metrics in chunking_metrics_by_strategy.items()
)
all_variants_comparison = (
    pd.DataFrame(all_variant_rows)
    .set_index("Вариант")
    .loc[
        [
            "TF-IDF",
            "paraphrase-multilingual-MiniLM-L12-v2",
            "multilingual-e5-small",
            "distiluse-base-multilingual-cased-v2",
            "E5 + fixed_64_overlap_16",
            "E5 + paragraph_aware_64",
        ]
    ]
)
all_variants_comparison.round(3)

,Recall@1,Recall@3,Recall@5,Recall@10,Precision@5,MRR
Вариант,,,,,,
TF-IDF,0.156,0.312,0.438,0.562,0.088,0.276
paraphrase-multilingual-MiniLM-L12-v2,0.344,0.469,0.719,0.781,0.144,0.471
multilingual-e5-small,0.406,0.562,0.625,0.750,0.125,0.506
distiluse-base-multilingual-cased-v2,0.250,0.438,0.469,0.531,0.094,0.356
E5 + fixed_64_overlap_16,0.406,0.750,0.875,0.969,0.175,0.609
E5 + paragraph_aware_64,0.438,0.812,0.875,0.969,0.175,0.650


In [21]:
def relevant_rank(results: Dict, query_id: int, relevant_docs: List[str]):
    retrieved = [item["filename"] for item in results.get(query_id, [])]
    matching_ranks = [
        rank for rank, filename in enumerate(retrieved, start=1)
        if filename in relevant_docs
    ]
    return min(matching_ranks) if matching_ranks else None


query_rank_rows = []
for query_data in test_queries:
    query_id = query_data["id"]
    relevant_docs = query_data["relevant_docs"]
    query_rank_rows.append(
        {
            "id": query_id,
            "query": query_data["query"],
            "relevant_document": relevant_docs[0],
            "TF-IDF rank": relevant_rank(tfidf_results, query_id, relevant_docs),
            "E5 whole-document rank": relevant_rank(
                semantic_results_by_model["multilingual-e5-small"],
                query_id,
                relevant_docs,
            ),
            "E5 fixed-window rank": relevant_rank(
                chunking_results_by_strategy["fixed_64_overlap_16"],
                query_id,
                relevant_docs,
            ),
            "E5 paragraph-aware rank": relevant_rank(
                chunking_results_by_strategy["paragraph_aware_64"],
                query_id,
                relevant_docs,
            ),
        }
    )

query_rank_comparison = pd.DataFrame(query_rank_rows)
query_rank_comparison

,id,query,relevant_document,TF-IDF rank,E5 whole-document rank,E5 fixed-window rank,E5 paragraph-aware rank
0,1,Какие этапы проходит исходный код C++ от препр...,Универ/5 семестр/Программирование на языке C++.md,NaN,1.0,4.0,2.0
1,2,"Где искать причину, если программа не линкуетс...",Универ/5 семестр/Программирование на языке C++.md,11.0,14.0,7.0,7.0
2,3,Почему макрос для квадрата неправильно обрабат...,Универ/5 семестр/Программирование на языке C++.md,NaN,NaN,NaN,NaN
3,4,Как передать аргументы командной строки в функ...,Универ/5 семестр/Программирование на языке C++.md,2.0,1.0,2.0,2.0
4,5,"Чем связаны понятия Data Science, Big Data и d...",Универ/5 семестр/Инструменты анализа данных.md,15.0,3.0,2.0,2.0
5,6,Какие три V описывают большие данные?,Универ/5 семестр/Инструменты анализа данных.md,NaN,NaN,2.0,3.0
6,7,"Почему алгоритм не спасет, если данные собраны...",Универ/5 семестр/Инструменты анализа данных.md,NaN,19.0,2.0,1.0
7,8,В каких случаях вместо машинного обучения дост...,Универ/5 семестр/Инструменты анализа данных.md,9.0,1.0,1.0,2.0
8,9,Какая метрика регрессии меньше реагирует на вы...,Универ/6 семестр/Машинное обучение.md,9.0,8.0,6.0,6.0
9,10,"Как интерпретировать коэффициент R-квадрат, ес...",Универ/6 семестр/Машинное обучение.md,18.0,NaN,5.0,2.0
